# 04 模型訓練

| | 內容 |
|---|---|
| 輸入 | `data/processed/`（01、03 的輸出） |
| 輸出 | `output/backtest/<時間>_notebook_04_tuning/`（逐窗分數、逐日 6 目標、432 點曲線 `curves.csv`，供 05 讀取）、`output/reports/04_runtimes.json` |

**現行模型**：
- 時刻（佔總分約 93%）：條件經驗分布（日別 × 夏月 × 星期 × 氣溫 2 箱）+ 學習式 LightGBM 在 PMF 層級修正 + 貝氏決策
- 量值：LightGBM 分位數迴歸（τ = 0.625 / 0.70 / 0.50）
- 氣象：五都 4 個溫度項；歷史用 CODiS 觀測，目標日用逐起點校正的 Accuweather
- 太陽光電：Windy 預測發電量 `pv_14`、`pv_sum`
- 6 目標 → 合成 432 點曲線

**回測設計**（`src/evaluation/backtest.py`）：每個窗從起點日 T 的次日 00:00 起算，負載截止於 T 23:50，
預測 3 天共 432 期；目標日的氣象只用預報（honest 模式）。調參組 = 現有 60 折 + 2025 年 9/19–10/9 每天起算的同季節窗，共 78 窗。
保留確認組（2026 年 7–9 月）只在選定最終設定後使用一次，本 notebook 不會去跑它。

observed 模式（目標日也用觀測，樂觀）只作對照，可由 `python main.py evaluate --weather-mode observed` 重現。

調參組回測約 15 分鐘。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "config").is_dir():   # 找到專案根目錄（notebook 可從任何位置執行）
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import os
import polars as pl
from src.logging_setup import setup_logging

os.chdir(ROOT)
setup_logging(stage="notebook_04", level="WARNING")
pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(30)
pl.Config.set_fmt_str_lengths(60)

polars.config.Config

In [2]:
import datetime as dt
import json
import time

from config import paths, settings
from src import workflow
from src.data import checks
from src.evaluation import backtest

runtimes = {}

## 1. 調參組回測（78 窗）

In [3]:
start = time.perf_counter()
run_dir = backtest.run("tuning", "notebook_04")
runtimes["backtest_tuning"] = round(time.perf_counter() - start, 1)
summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
paths.REPORTS_DIR.mkdir(parents=True, exist_ok=True)
{k: summary[k] for k in ("n_windows", "skipped", "manifest_sha256", "data_matches_manifest", "runtime_seconds")}

2026-09-30 20:15:45 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-09-30 20:15:46 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-09-30 20:15:47 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-09-30 20:15:52 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-09-30 20:15:54 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


...

2026-09-30 20:30:24 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-09-30 20:30:27 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-09-30 20:30:29 | WARNING  | src.workflow | total_score = 1.04169（標準誤 0.06788），最壞折 2.77256（2025-08-01）［氣象模式 honest］
shape: (5, 5)
┌─────────────────┬──────────┬────────┬──────────────┬──────────┐
│ component       ┆ raw      ┆ weight ┆ contribution ┆ share    │
│ ---             ┆ ---      ┆ ---    ┆ ---          ┆ ---      │
│ str             ┆ f64      ┆ f64    ┆ f64          ┆ f64      │
╞═════════════════╪══════════╪════════╪══════════════╪══════════╡
│ s_peak_time     ┆ 6.487846 ┆ 0.15   ┆ 0.973177     ┆ 0.934226 │
│ s_ramp_up       ┆ 0.173534 ┆ 0.15   ┆ 0.02603      ┆ 0.024988 │
│ s_ramp_down     ┆ 0.148411 ┆ 0.1    ┆ 0.014841     ┆ 0.014247 │
│ s_peak_mw       ┆ 0.024205 ┆ 0.6    ┆ 0.014523     ┆ 0.013942 │
│ s_under_penalty ┆ 0.013122 ┆ 1.0    ┆ 0.013122     ┆ 0.012597 │
└─────────────────┴──────────┴────────┴──────────────┴──────────┘


2026-09-30 20:30:29 | WARNING  | src.evaluation.backtest | 回測 [notebook_04／tuning] 78 窗：total_score 1.04169（標準誤 0.06788），同季節 0.76025 → 20260930_201544_notebook_04_tuning


{'n_windows': 78,
 'skipped': [],
 'manifest_sha256': 'ecda90d93d378091edd4cbb09e7f42fca63a2a97447c50045a3fa47b6466ab49',
 'data_matches_manifest': True,
 'runtime_seconds': 887.5}

In [4]:
pl.DataFrame([{"範圍": "全體", **summary["overall"]}]
             + [{"範圍": name, **value} for name, value in summary["subsets"].items()])

範圍,n,total_score,stderr,worst,worst_origin,s_peak_mw,s_peak_time,s_ramp_up,s_ramp_down,s_under_penalty
str,i64,f64,f64,f64,str,f64,f64,f64,f64,f64
"""全體""",78,1.041693,0.06788,2.772556,"""2025-08-01""",0.024205,6.487846,0.173534,0.148411,0.013122
"""同季節""",21,0.760246,0.092808,1.834123,"""2025-09-20""",0.02478,4.642609,0.177568,0.149705,0.007381
"""週四五六""",11,1.185827,0.225021,2.414555,"""2025-06-11""",0.021575,7.500719,0.162716,0.137515,0.009615
"""含週六""",35,1.049308,0.111982,2.772556,"""2025-08-01""",0.020797,6.526501,0.195866,0.164591,0.012016
"""夏月末期""",18,0.622213,0.059315,1.02721,"""2025-09-27""",0.022799,3.707244,0.196415,0.163904,0.006594
"""含連假或停班""",27,1.033285,0.095098,1.908072,"""2026-04-04""",0.027606,6.348033,0.250477,0.182542,0.008691


## 2. 以鎖定設定重新訓練並預測（開發期：截止日次日起 3 天）

與比賽當天 `run_submission.py` 第 5–6 步是同一個函式。

In [5]:
origin = checks.cutoff_date()
days = [origin + dt.timedelta(days=k) for k in range(1, settings.PREDICT_HORIZON_DAYS + 1)]
start = time.perf_counter()
curves, intended, requested = workflow.predict_days(origin, days)
runtimes["train_and_predict"] = round(time.perf_counter() - start, 1)
pl.DataFrame([
    {"日期": str(day), "p_day": t.p_day, "t_day": t.t_day, "p_night": t.p_night,
     "t_night": t.t_night, "ramp_up": t.ramp_up, "ramp_down": t.ramp_down}
    for day, t in intended.items()
])

2026-09-30 20:30:30 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-09-30 20:30:31 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-09-30 20:30:32 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-09-30 20:30:33 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-09-30 20:30:41 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-09-30 20:30:44 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-09-30 20:30:46 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


日期,p_day,t_day,p_night,t_night,ramp_up,ramp_down
str,f64,i64,f64,i64,f64,f64
"""2026-07-01""",39766.857637,840,36729.72771,1030,1426.894147,716.383084
"""2026-07-02""",39905.866259,840,36456.997301,1030,1482.205991,693.389594
"""2026-07-03""",39491.71789,840,36574.059603,1030,1463.318588,672.770172


In [6]:
(paths.REPORTS_DIR / "04_runtimes.json").write_text(json.dumps(runtimes, indent=2), encoding="utf-8")
runtimes

{'backtest_tuning': 887.5, 'train_and_predict': 19.3}

## 3. 實驗總表

每個實驗都走同一套回測（調參組 78 窗、honest 模式），與當時的參考配對比較。
**所有變體都在執行前決定**，看到結果後沒有追加或微調，以免對調參組過度配適。

選模門檻：全部調參窗的配對改善大於 1 個標準誤，且同季節子集沒有變差超過 1 個標準誤；
通過者再以另外 2 個種子確認。差值為正代表變差；「對照」是配對比較的參考紀錄。

In [7]:
EXPERIMENTS = {
    "current_best": "參考：含 UV、不含 Windy",
    "codis_lag1": "CODiS 只到起點前一天",
    "4a_A1_noUV": "氣象不含 UV（採用）",
    "4a_A1raw_noUV_uncal": "目標日預報不校正",
    "4a_B_forecast": "歷史也改用預報",
    "4a_N_noweather": "不用氣象",
    "4c_windy": "加入 Windy 太陽光電預測（採用）",
    "4e_E1_timing_seeds5": "時刻學習模型 5 種子平均",
    "4f_drop_zero_split": "刪除 6 個未使用的特徵（採用）",
    "4g_mix03": "時刻學習權重 0.3",
    "4g_mix07": "時刻學習權重 0.7",
    "4h_magnitude_seeds5": "量值模型 5 種子平均",
    "4d_recursive_ar": "對照：10 分鐘一步的遞迴 AR",
}
names = pl.DataFrame({"label": list(EXPERIMENTS), "實驗": list(EXPERIMENTS.values())})
experiments = names.join(backtest.experiment_table("2026*"), on="label", how="left")
experiments.select("實驗", "total_score", "同季節", "s_peak_time", "差值", "標準誤倍數",
                   "同季節差值", "通過門檻", "對照", "耗時秒")

實驗,total_score,同季節,s_peak_time,差值,標準誤倍數,同季節差值,通過門檻,對照,耗時秒
str,f64,f64,f64,f64,f64,f64,bool,str,f64
"""參考：含 UV、不含 Windy""",1.056294,0.758808,6.580069,null,null,null,null,null,912.1
"""CODiS 只到起點前一天""",1.059357,0.759717,6.600641,0.003063,0.672629,0.000909,false,"""20260927_142000_current_best_tuning""",1482.0
"""氣象不含 UV（採用）""",1.064475,0.760738,6.629968,0.008181,0.807855,0.00193,false,"""20260927_142000_current_best_tuning""",1919.0
"""目標日預報不校正""",1.080949,0.777722,6.744901,0.024655,1.937746,0.018914,false,"""20260927_142000_current_best_tuning""",1975.8
"""歷史也改用預報""",1.076029,0.769114,6.715221,0.019735,1.580864,0.010306,false,"""20260927_142000_current_best_tuning""",2793.4
"""不用氣象""",1.087295,0.799477,6.773392,0.031002,1.911301,0.040669,false,"""20260927_142000_current_best_tuning""",1852.5
"""加入 Windy 太陽光電預測（採用）""",1.041693,0.760246,6.487846,-0.022781,1.810511,-0.000491,true,"""20260928_101300_4a_A1_noUV_tuning""",2097.8
"""時刻學習模型 5 種子平均""",1.059312,0.773529,6.60019,0.003018,0.620476,0.014721,false,"""20260927_142000_current_best_tuning""",4314.5
"""刪除 6 個未使用的特徵（採用）""",1.064475,0.760738,6.629968,0.0,inf,0.0,false,"""20260928_101300_4a_A1_noUV_tuning""",2097.1


In [8]:
experiments.select("實驗", "label", "覆寫")

實驗,label,覆寫
str,str,str
"""參考：含 UV、不含 Windy""","""current_best""","""{}"""
"""CODiS 只到起點前一天""","""codis_lag1""","""{}"""
"""氣象不含 UV（採用）""","""4a_A1_noUV""","""{""WEATHER_METRICS"": [""day_tmax"", ""day_tmin"", ""night_tmax"", ""…"
"""目標日預報不校正""","""4a_A1raw_noUV_uncal""","""{""WEATHER_METRICS"": [""day_tmax"", ""day_tmin"", ""night_tmax"", ""…"
"""歷史也改用預報""","""4a_B_forecast""","""{}"""
"""不用氣象""","""4a_N_noweather""","""{""ENABLE_TIER1_WEATHER"": false, ""TIMING_TEMPERATURE_BINS"": 1…"
"""加入 Windy 太陽光電預測（採用）""","""4c_windy""","""{""ENABLE_WINDY"": true}"""
"""時刻學習模型 5 種子平均""","""4e_E1_timing_seeds5""","""{""TIMING_LEARNED_N_SEEDS"": 5}"""
"""刪除 6 個未使用的特徵（採用）""","""4f_drop_zero_split""","""{""EXCLUDED_FEATURES"": [""is_lunar_new_year_eve"", ""night_price…"
